# AgroFlow no Google Colab (sojatech_local)

Roda o trabalho inteiro sem nenhum serviço pago: treina o classificador de folhas, extrai as notas fiscais, classifica as folhas de teste, gera a avaliação (tabelas, gráficos, LaTeX) e mostra o pipeline orientado a eventos funcionando.

Use um ambiente com **GPU T4** (Ambiente de execução → Alterar tipo → T4). Em CPU funciona, só mais devagar.

## 0. Preparar o ambiente

O repositório é privado: crie um token no GitHub (Settings → Developer settings → Fine-grained tokens, só leitura de *Contents* do repositório `agroflow`) e salve-o nos **Secrets do Colab** (ícone de chave na barra lateral) com o nome `GITHUB_TOKEN`, liberando o acesso para este notebook. Se o repositório for público, pule o token.

In [ ]:
import os, re, subprocess

REPO = "AbnerMSaconi/agroflow"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("/content/agroflow"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/agroflow"], check=True)
%cd /content/agroflow/sojatech_local
!pip install -q -r requirements.txt
!nvidia-smi -L || echo "sem GPU: roda em CPU, mais devagar"

In [ ]:
from IPython.display import Image, Markdown, display


def mostra_figuras(*nomes):
    for n in nomes:
        caminho = f"results/figures/{n}.png"
        if os.path.exists(caminho):
            display(Image(caminho, width=520))

## 1. Treino, extração e avaliação

Os dados já estão no repositório (`data/` na raiz): 40 notas de desenvolvimento, 40 de teste (com um layout que o extrator nunca viu) e 100 folhas de teste. O modelo não está versionado, então o primeiro passo é treiná-lo (~15 s numa T4).

In [ ]:
!python scripts/train_leaf_classifier.py
!python scripts/run_batch.py notas | tail -2
!python scripts/run_batch.py folhas | tail -1
!python scripts/evaluate.py > /dev/null
display(Markdown(open("results/resumo.md", encoding="utf-8").read()))
mostra_figuras("notas_acerto_vs_confidence_teste", "folhas_matriz_confusao", "folhas_calibracao")

## 2. Pipeline orientado a eventos

Copia uma nota, uma folha e um arquivo inválido para `bucket/entrada/` e roda o watcher uma vez (`--once`). Os válidos vão para `bucket/processados/` com o resultado gravado na tabela; o inválido vai para `bucket/rejeitados/` com o motivo em `bucket/rejeitados.log`. Na máquina local, `python scripts/watch_bucket.py` sem `--once` fica escutando a pasta.

In [ ]:
!mkdir -p bucket/entrada/notas bucket/entrada/imagens
!cp ../data/invoices/teste/pdf/nota_001.pdf bucket/entrada/notas/demo_nota.pdf
!cp ../data/leaves/test/ferrugem_comum/ferrugem_comum_000.jpg bucket/entrada/imagens/demo_folha.jpg
!echo "não é uma nota" > bucket/entrada/notas/demo_invalido.txt
!python scripts/watch_bucket.py --once
!echo; echo "rejeitados.log:"; cat bucket/rejeitados.log

## 3. Baixar os resultados

Baixa um zip com `results/` e as tabelas e figuras do relatório para você copiar para o repositório local e fazer o commit.

In [ ]:
!zip -qr /content/agroflow_resultados.zip results ../report/figuras ../report/tabelas -x "results/agroflow.db"
from google.colab import files
files.download("/content/agroflow_resultados.zip")